In [ ]:
#Using an MLP to detect strikefoot and run corrections on the LSTM's prediction
import json
import pandas as pd

In [ ]:
with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/strikefoot_keypoints.json", "r") as file:
    strikefoot_kpts = json.load(file)

with open("/Users/abhinavarora/Desktop/CadenceCV/ml/data/non_strikefoot_keypoints.json", "r") as file:
    non_strikefoot_kpts = json.load(file)

In [ ]:
#Restructuring the keypoints to flatten out the dict entries in the json file
def flatten_kpts(kpts):
    res = []
    for frame in kpts:
        d = {}
        d["video"] = frame["video"]
        for (index, k) in enumerate(frame["Keypoints"]):
            d["keypoint " + str(index)] = k
        d["Frame"] = frame["Frame"]
    
    return res

strikefoot_kpts = flatten_kpts(strikefoot_kpts)
non_strikefoot_kpts = flatten_kpts(non_strikefoot_kpts)

In [ ]:
strikefoot_df = pd.DataFrame(strikefoot_kpts)
non_strikefoot_df = pd.DataFrame(non_strikefoot_kpts)

non_strikefoot_df = non_strikefoot_df.dropna(axis=0).reset_index(drop=True)
strikefoot_df = strikefoot_df.dropna(axis=0).reset_index(drop=True)

#Dropping the frames of non strikefoot df that are in strikefoot df
non_strikefoot_df = non_strikefoot_df[~(strikefoot_df["Frames"].isin(non_strikefoot_df["Frames"]))]
non_strikefoot_df.reset_index(drop=True)
strikefoot_df.reset_index(drop=True)

#Sorting by frame and side (but for the same videos)
def sort_by_frame_and_side(df:pd.DataFrame):
    vids = df["video"].unique()
    dfs = []
    for v in vids:
        d:pd.Series = df[df["video"] == v]
        d = d.sort_values(["Frame"], axis=0)
        dfs.append(d)
    
    res = pd.DataFrame(pd.concat(dfs, axis=0)).reset_index(drop=True)
    return res

non_strikefoot_df = sort_by_frame_and_side(non_strikefoot_df)
strikefoot_df = sort_by_frame_and_side(strikefoot_df)

